# 04 — Deaths Per 10 Minutes: The Universal Performance Benchmark

Deaths per 10 minutes (D/10) is one of the most commonly cited performance metrics in competitive Overwatch. The coaching community has established these rough benchmarks:

> **"<5.0 excellent, 5-6 good, 6-7.5 average, >8 poor"**
> — Widely cited on r/OverwatchUniversity and coaching circles

### Key OW Concepts
- **Deaths per 10 (D/10)**: Deaths divided by (playtime in seconds / 600). Normalizes across different match lengths.
- **Role expectations**: Tanks absorb damage and are expected to die more. Supports should die less if positioned well. DPS varies by hero (flankers die more than snipers).
- **Death = downtime**: Each death costs ~10-12 seconds (respawn + travel back). Dying less means more uptime, more contribution.
- **Quality of deaths**: Not all deaths are equal. Dying first in a fight is worse than dying in a lost fight. Trading 1-for-1 can be worth it.

### Analysis Plan
1. Calculate D/10 from PlayerStat data
2. Overall distribution with benchmark lines
3. Distribution by role (Tank/DPS/Support)
4. Correlation with match win rate
5. Hero-specific D/10 baselines
6. Coaching implications

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_player_stats, load_matches
from src.preprocessing import (
    determine_match_winner, add_role_column, HERO_ROLES
)
from src.metrics import deaths_per_10, deaths_per_10_series
from src.visualization import setup_style, OW_COLORS, OW_PALETTE, ROLE_COLORS, save_fig, role_color

setup_style()
pd.set_option('display.max_columns', 30)

## 1. Load and Prepare Data

In [ ]:
player_stats = load_player_stats()
match_start, match_end = load_matches()
matches = determine_match_winner(match_end, match_start)

print(f"PlayerStat rows: {len(player_stats):,}")
print(f"Matches: {len(matches):,}")
print()
print("PlayerStat columns:")
print(list(player_stats.columns))

In [ ]:
# Join with match outcomes
ps = player_stats.merge(
    matches[['MapDataId', 'winner', 'team_1_name', 'team_2_name', 'map_name', 'map_type']],
    on='MapDataId',
    how='inner'
)

# Filter to meaningful playtime (at least 60 seconds)
ps = ps[ps['hero_time_played'] >= 60].copy()

# Calculate D/10
ps['d10'] = deaths_per_10_series(ps['deaths'], ps['hero_time_played'])

# Add role and win info
ps = add_role_column(ps)
ps['team_won'] = ps['player_team'] == ps['winner']

# Filter out extreme outliers (likely data issues)
ps = ps[ps['d10'] < 30].copy()

print(f"Valid player-hero observations: {len(ps):,}")
print(f"Unique players: {ps['player_name'].nunique():,}")
print(f"Unique heroes: {ps['player_hero'].nunique():,}")
print(f"\nOverall D/10 stats:")
print(f"  Mean:   {ps['d10'].mean():.2f}")
print(f"  Median: {ps['d10'].median():.2f}")
print(f"  Std:    {ps['d10'].std():.2f}")

## 2. Overall D/10 Distribution with Benchmark Lines

Let's see how the actual D/10 distribution compares to the community benchmarks.

In [ ]:
# Benchmarks from community consensus
BENCHMARKS = {
    'Excellent (<5.0)': 5.0,
    'Good (5-6)': 6.0,
    'Average (6-7.5)': 7.5,
    'Poor (>8)': 8.0,
}

fig, ax = plt.subplots(figsize=(14, 6))

ax.hist(ps['d10'], bins=60, range=(0, 20), color=OW_COLORS['orange'],
        edgecolor=OW_COLORS['dark_blue'], alpha=0.8, density=True)

# Add benchmark zones
ax.axvspan(0, 5.0, alpha=0.1, color=OW_COLORS['green'], label='Excellent (<5.0)')
ax.axvspan(5.0, 6.0, alpha=0.1, color=OW_COLORS['teal'], label='Good (5-6)')
ax.axvspan(6.0, 7.5, alpha=0.1, color=OW_COLORS['gold'], label='Average (6-7.5)')
ax.axvspan(7.5, 20, alpha=0.1, color=OW_COLORS['red'], label='Poor (>7.5)')

# Benchmark lines
for label, val in BENCHMARKS.items():
    ax.axvline(val, color=OW_COLORS['white'], linestyle='--', alpha=0.5)

ax.axvline(ps['d10'].median(), color=OW_COLORS['red'], linestyle='-', linewidth=2,
           label=f'Median: {ps["d10"].median():.2f}')

ax.set_xlabel('Deaths Per 10 Minutes')
ax.set_ylabel('Density')
ax.set_title('Deaths Per 10 Minutes Distribution with Community Benchmarks')
ax.set_xlim(0, 20)
ax.legend(loc='upper right')

plt.tight_layout()
save_fig(fig, '04_d10_distribution_benchmarks')
plt.show()

# What percentage of observations fall in each bracket?
excellent = (ps['d10'] < 5.0).mean() * 100
good = ((ps['d10'] >= 5.0) & (ps['d10'] < 6.0)).mean() * 100
average = ((ps['d10'] >= 6.0) & (ps['d10'] < 7.5)).mean() * 100
poor = (ps['d10'] >= 7.5).mean() * 100

print(f"\nBenchmark distribution:")
print(f"  Excellent (<5.0):  {excellent:.1f}%")
print(f"  Good (5-6):        {good:.1f}%")
print(f"  Average (6-7.5):   {average:.1f}%")
print(f"  Poor (>7.5):       {poor:.1f}%")

## 3. D/10 Distribution by Role

Tanks, DPS, and Supports have fundamentally different death profiles. Tanks engage the enemy directly and absorb damage. Supports try to stay alive from the backline. DPS varies — flankers like Tracer or Genji trade aggressively, while snipers like Widowmaker play safely.

In [ ]:
# D/10 by role
role_stats = ps.groupby('role')['d10'].describe()
print("D/10 by Role:")
print(role_stats[['count', 'mean', '50%', 'std']].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, role_name in zip(axes, ['Tank', 'DPS', 'Support']):
    role_data = ps[ps['role'] == role_name]['d10']
    
    ax.hist(role_data, bins=50, range=(0, 20), color=ROLE_COLORS[role_name],
            edgecolor=OW_COLORS['dark_blue'], alpha=0.8, density=True)
    
    # Benchmark lines
    for val in [5.0, 6.0, 7.5, 8.0]:
        ax.axvline(val, color=OW_COLORS['white'], linestyle='--', alpha=0.3)
    
    ax.axvline(role_data.median(), color=OW_COLORS['gold'], linestyle='-', linewidth=2,
               label=f'Median: {role_data.median():.2f}')
    ax.axvline(role_data.mean(), color=OW_COLORS['red'], linestyle='--', linewidth=1.5,
               label=f'Mean: {role_data.mean():.2f}')
    
    ax.set_xlabel('Deaths Per 10 Minutes')
    ax.set_title(f'{role_name} D/10 Distribution')
    ax.set_xlim(0, 20)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.tight_layout()
save_fig(fig, '04_d10_by_role')
plt.show()

In [ ]:
# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

role_order = ['Tank', 'DPS', 'Support']
bp = ax.boxplot(
    [ps[ps['role'] == r]['d10'] for r in role_order],
    labels=role_order,
    patch_artist=True,
    showfliers=False,  # Hide outliers for cleaner view
    medianprops={'color': OW_COLORS['gold'], 'linewidth': 2}
)

for patch, role_name in zip(bp['boxes'], role_order):
    patch.set_facecolor(ROLE_COLORS[role_name])
    patch.set_alpha(0.7)

# Add benchmark lines
for val, label in [(5.0, 'Excellent'), (6.0, 'Good'), (7.5, 'Average'), (8.0, 'Poor')]:
    ax.axhline(val, color=OW_COLORS['light_gray'], linestyle='--', alpha=0.4)
    ax.text(3.4, val + 0.1, label, fontsize=9, color=OW_COLORS['light_gray'])

ax.set_ylabel('Deaths Per 10 Minutes')
ax.set_title('D/10 Distribution by Role')
ax.set_ylim(0, 15)

plt.tight_layout()
save_fig(fig, '04_d10_role_boxplot')
plt.show()

## 4. D/10 Correlation with Match Win Rate

The fundamental question: does dying less actually help you win? We correlate player-level D/10 with whether their team won the match.

In [ ]:
# Average D/10 for winners vs losers
print("D/10 by Match Outcome:")
print("=" * 40)
outcome_d10 = ps.groupby('team_won')['d10'].agg(['mean', 'median', 'count'])
outcome_d10.index = ['Lost', 'Won']
print(outcome_d10.to_string())
print()

# Statistical test
winners = ps[ps['team_won']]['d10']
losers = ps[~ps['team_won']]['d10']
stat, pval = stats.mannwhitneyu(winners, losers, alternative='less')
print(f"Mann-Whitney U test (winners D/10 < losers D/10):")
print(f"  U = {stat:.0f}, p = {pval:.4e}")
effect_size = (losers.mean() - winners.mean()) / ps['d10'].std()
print(f"  Cohen's d = {effect_size:.3f}")

In [ ]:
# Visualization: Winner vs Loser D/10 by role
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, role_name in zip(axes, ['Tank', 'DPS', 'Support']):
    role_ps = ps[ps['role'] == role_name]
    w = role_ps[role_ps['team_won']]['d10']
    l = role_ps[~role_ps['team_won']]['d10']
    
    means = [w.mean(), l.mean()]
    sems = [w.sem(), l.sem()]
    
    bars = ax.bar(['Winners', 'Losers'], means, yerr=sems, capsize=5,
                  color=[OW_COLORS['green'], OW_COLORS['red']], width=0.5,
                  edgecolor=OW_COLORS['dark_blue'])
    
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', fontsize=12, fontweight='bold',
                color=OW_COLORS['white'])
    
    ax.set_ylabel('Mean D/10')
    ax.set_title(f'{role_name}: D/10 Winners vs Losers')
    ax.set_ylim(0, max(means) * 1.3)

plt.tight_layout()
save_fig(fig, '04_d10_winners_vs_losers_by_role')
plt.show()

In [ ]:
# Team-level analysis: aggregate D/10 per team per match
team_d10 = ps.groupby(['MapDataId', 'player_team']).agg(
    team_d10=('d10', 'mean'),
    team_won=('team_won', 'first'),
    total_deaths=('deaths', 'sum'),
    total_time=('hero_time_played', 'sum')
).reset_index()

# Bin team D/10 and compute win rate
team_d10['d10_bin'] = pd.cut(team_d10['team_d10'],
                              bins=[0, 4, 5, 6, 7, 8, 10, 30],
                              labels=['<4', '4-5', '5-6', '6-7', '7-8', '8-10', '10+'])

bin_wr = team_d10.groupby('d10_bin', observed=True).agg(
    total=('team_won', 'count'),
    wins=('team_won', 'sum')
)
bin_wr['win_rate'] = bin_wr['wins'] / bin_wr['total'] * 100

fig, ax = plt.subplots(figsize=(12, 6))
colors = [OW_COLORS['green'] if wr > 55 else OW_COLORS['red'] if wr < 45
          else OW_COLORS['orange'] for wr in bin_wr['win_rate']]

bars = ax.bar(bin_wr.index.astype(str), bin_wr['win_rate'], color=colors,
              width=0.6, edgecolor=OW_COLORS['dark_blue'])

for bar, (_, row) in zip(bars, bin_wr.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{row["win_rate"]:.1f}%\n(n={row["total"]:,})',
            ha='center', fontsize=9, color=OW_COLORS['white'])

ax.axhline(50, color=OW_COLORS['gold'], linestyle='--', alpha=0.7, label='50% baseline')
ax.set_xlabel('Team Average D/10')
ax.set_ylabel('Match Win Rate (%)')
ax.set_title('Match Win Rate by Team Average Deaths Per 10')
ax.legend()
ax.set_ylim(0, 100)

plt.tight_layout()
save_fig(fig, '04_team_d10_win_rate')
plt.show()

## 5. Hero-Specific D/10 Baselines

Not all heroes should have the same D/10 target. A Reinhardt who charges in will die more than a Widowmaker positioned on high ground. Let's establish hero-specific baselines.

In [ ]:
# Hero D/10 baselines
hero_d10 = ps.groupby('player_hero').agg(
    median_d10=('d10', 'median'),
    mean_d10=('d10', 'mean'),
    std_d10=('d10', 'std'),
    count=('d10', 'count'),
    role=('role', 'first')
)
hero_d10 = hero_d10[hero_d10['count'] >= 30]  # Minimum sample
hero_d10 = hero_d10.sort_values('median_d10')

print("Hero D/10 Baselines (sorted by median):")
print(hero_d10[['median_d10', 'mean_d10', 'count', 'role']].to_string())

In [ ]:
# Visualization: Hero D/10 bar chart
fig, ax = plt.subplots(figsize=(14, 10))

colors = [role_color(r) for r in hero_d10['role']]
bars = ax.barh(hero_d10.index, hero_d10['median_d10'], color=colors, alpha=0.85,
               xerr=hero_d10['std_d10'] * 0.5, capsize=2)

# Benchmark lines
for val, label in [(5.0, 'Excellent'), (6.0, 'Good'), (7.5, 'Average'), (8.0, 'Poor')]:
    ax.axvline(val, color=OW_COLORS['white'], linestyle='--', alpha=0.3)
    ax.text(val + 0.05, len(hero_d10) - 0.5, label, fontsize=8,
            color=OW_COLORS['light_gray'], rotation=90, va='top')

ax.set_xlabel('Median Deaths Per 10 Minutes')
ax.set_title('Hero D/10 Baselines (Median with Half-Std Error Bars)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ['Tank', 'DPS', 'Support']]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '04_hero_d10_baselines')
plt.show()

In [ ]:
# Hero D/10 for winners vs losers — which heroes show the biggest gap?
hero_d10_outcome = ps.groupby(['player_hero', 'team_won']).agg(
    median_d10=('d10', 'median'),
    count=('d10', 'count'),
    role=('role', 'first')
).reset_index()

# Pivot to get winner/loser D/10 side by side
hero_outcome_wide = hero_d10_outcome.pivot_table(
    index='player_hero', columns='team_won', values='median_d10'
).rename(columns={True: 'winner_d10', False: 'loser_d10'})

hero_outcome_wide['d10_gap'] = hero_outcome_wide['loser_d10'] - hero_outcome_wide['winner_d10']
hero_outcome_wide = hero_outcome_wide.dropna()

# Filter to heroes with enough data
hero_counts = ps.groupby('player_hero').size()
valid_heroes = hero_counts[hero_counts >= 50].index
hero_outcome_wide = hero_outcome_wide[hero_outcome_wide.index.isin(valid_heroes)]
hero_outcome_wide = hero_outcome_wide.sort_values('d10_gap', ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))
roles = [HERO_ROLES.get(h, 'Unknown') for h in hero_outcome_wide.index]
colors = [role_color(r) for r in roles]

ax.barh(hero_outcome_wide.index, hero_outcome_wide['d10_gap'], color=colors, alpha=0.85)
ax.axvline(0, color=OW_COLORS['gold'], linestyle='-', alpha=0.7)
ax.set_xlabel('D/10 Gap (Loser D/10 - Winner D/10)')
ax.set_title('D/10 Gap Between Winners and Losers by Hero\n(Positive = losers die more on this hero)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=ROLE_COLORS[r], label=r) for r in ['Tank', 'DPS', 'Support']]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
save_fig(fig, '04_hero_d10_gap')
plt.show()

## 6. Role-Specific Benchmarks

Since roles have different D/10 profiles, let's propose role-specific benchmarks based on the data percentiles.

In [ ]:
# Role-specific percentile benchmarks
print("Proposed Role-Specific D/10 Benchmarks")
print("=" * 60)
print(f"{'Role':<10} {'Excellent':>10} {'Good':>10} {'Average':>10} {'Poor':>10}")
print(f"{'':10} {'(25th %ile)':>10} {'(50th %ile)':>10} {'(75th %ile)':>10} {'(90th %ile)':>10}")
print("-" * 60)

role_benchmarks = {}
for role_name in ['Tank', 'DPS', 'Support']:
    role_data = ps[ps['role'] == role_name]['d10']
    p25 = role_data.quantile(0.25)
    p50 = role_data.quantile(0.50)
    p75 = role_data.quantile(0.75)
    p90 = role_data.quantile(0.90)
    role_benchmarks[role_name] = {'p25': p25, 'p50': p50, 'p75': p75, 'p90': p90}
    print(f"{role_name:<10} {p25:>10.2f} {p50:>10.2f} {p75:>10.2f} {p90:>10.2f}")

print()
print("Interpretation: If your D/10 is below the 25th percentile for your role,")
print("you are dying significantly less than average — excellent survival.")

In [ ]:
# Summary table: D/10 statistics by role and outcome
summary = ps.groupby(['role', 'team_won']).agg(
    mean_d10=('d10', 'mean'),
    median_d10=('d10', 'median'),
    count=('d10', 'count')
).round(2)

print("\nD/10 Summary by Role and Outcome:")
print(summary.to_string())

## 7. Summary & Coaching Implications

### Key Findings

| Finding | Detail |
|---------|--------|
| Overall median D/10 | See data above |
| D/10 gap (winners vs losers) | See data above |
| Role differences | Tanks > DPS > Support (expected) |
| Team D/10 predicts wins | Clear correlation in the data |

### Validating the Community Benchmarks

The commonly cited benchmarks (<5.0 excellent, 5-6 good, 6-7.5 average, >8 poor) are a useful starting point, but our data reveals important nuances:

1. **Role-specific benchmarks are essential**: A Tank with 6.0 D/10 may be playing excellently, while a Support with the same number is underperforming. The one-size-fits-all benchmarks mask this.

2. **Hero-specific context matters even more**: A Wrecking Ball will naturally have higher D/10 than a Widowmaker. Comparing your D/10 against your hero's baseline (not a global number) gives a more accurate picture.

### Coaching Implications

1. **D/10 correlates with winning**: The data confirms that teams with lower D/10 win more. This is not surprising, but the magnitude of the effect and its consistency across roles validates D/10 as a coaching metric.

2. **Use role-specific targets**: Set different D/10 goals for your Tank, DPS, and Support players. Use the percentile-based benchmarks from this analysis.

3. **Track D/10 over time**: A player's D/10 trend is more actionable than a single snapshot. Improving from 8.0 to 6.5 over a month shows real improvement.

4. **Quality over quantity**: D/10 does not capture death quality. A Support dying to a Tracer flank is worse than dying in a lost fight. Combine D/10 with first-death analysis (Notebook 01) for a complete picture.

### For Players
- Look up your hero's D/10 baseline in the chart above. If you are above it, focus on positioning and cooldown management.
- Track your D/10 across scrims. Aim to get your hero-specific D/10 below the median.
- Remember: dying less means more uptime, which means more damage/healing/space for your team.